In [ ]:
!pip install --upgrade --quiet keras
!pip install --upgrade --quiet opencv-python

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import glob
import cv2
from tqdm.notebook import tqdm
import time
import random

import jax
import jax.ops as ops
import jax.numpy as jnp
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P
from jax.experimental import mesh_utils
from jax import checkpoint
import keras
import keras.layers as layers
import keras.optimizers as optimizers
import keras.ops as ops
#import keras_hub
from concurrent.futures import ThreadPoolExecutor, as_completed
#import albumentations as A
import time
import kagglehub

In [ ]:
print(f"keras version: {keras.__version__}")
print(f"jax version: {jax.__version__}")

In [ ]:
data_path = "/kaggle/input/datasets/jobayerhossain/urban-model-2-processed-data/model_2"

In [ ]:
events = glob.glob(f"{data_path}/event_*")
data_not_loaded = True

In [ ]:
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    keras.utils.set_random_seed(seed)
    #jax.config.update("jax_default_prng_impl", "rbg")


In [ ]:
def load_event_data(event, mmap_mode="r"):
    data = {}

    # --- 1. Load 1D Features (Memory Mapped & Explicit) ---
    data["1d_water_level"] = np.load(f"{event}/1d_water_level.npy", mmap_mode=mmap_mode)
    data["inlet_flow"] = np.load(f"{event}/inlet_flow.npy", mmap_mode=mmap_mode)


    data["flow_from_node_1d"] = np.load(f"{event}/flow_from_node_1d.npy", mmap_mode=mmap_mode)
    data["velocity_from_node_1d"] = np.load(f"{event}/velocity_from_node_1d.npy", mmap_mode=mmap_mode)
    data["flow_to_node_1d"] = np.load(f"{event}/flow_to_node_1d.npy", mmap_mode='r')
    data["velocity_to_node_1d"] = np.load(f"{event}/velocity_to_node_1d.npy", mmap_mode=mmap_mode)

    # --- 2. Load 2D Features (Memory Mapped & Explicit) ---
    data["rainfall"] = np.load(f"{event}/rainfall.npy", mmap_mode=mmap_mode)
    data["water_level"] = np.load(f"{event}/water_level.npy", mmap_mode=mmap_mode)
    data["water_volume"] = np.load(f"{event}/water_volume.npy", mmap_mode=mmap_mode)

    # Edge features - From Node
    data["flow_from_node_2d"] = np.load(f"{event}/flow_from_node_2d.npy", mmap_mode=mmap_mode)
    data["velocity_from_node_2d"] = np.load(f"{event}/velocity_from_node_2d.npy", mmap_mode=mmap_mode)

    # Edge features - To Node
    data["flow_to_node_2d"] = np.load(f"{event}/flow_to_node_2d.npy", mmap_mode=mmap_mode)
    data["velocity_to_node_2d"] = np.load(f"{event}/velocity_to_node_2d.npy", mmap_mode=mmap_mode)
    return data

In [ ]:
def load_static_feature(data_path):
    # Load raw files
    edges_1d_static = pd.read_csv(f"{data_path}/edges_1d_static.csv")
    nodes_1d_static = pd.read_csv(f"{data_path}/nodes_1d_static.csv")
    edges_2d_static = pd.read_csv(f"{data_path}/edges_2d_static.csv")
    nodes_2d_static = pd.read_csv(f"{data_path}/nodes_2d_static.csv")
    connection = pd.read_csv(f"{data_path}/1d2d_connection.csv")
    edge_index_2d = pd.read_csv(f"{data_path}/edge_index_2d.csv")
    edge_index_1d = pd.read_csv(f"{data_path}/edge_index_1d.csv")

    # Correcting the merges
    edges_1d_static = edges_1d_static.merge(edge_index_1d, on="edge_idx")
    edges_2d_static = edges_2d_static.merge(edge_index_2d, on="edge_idx")

    # Calculate max edges for padding (must match the dynamic data shapes)
    max_e_2d_from = edge_index_2d.groupby("from_node").size().max()
    max_e_2d_to = edge_index_2d.groupby("to_node").size().max()
    max_e_1d_from = edge_index_1d.groupby("from_node").size().max()
    max_e_1d_to = edge_index_1d.groupby("to_node").size().max()

    def pad_list(l, max_len):
        return l + [0.0] * (max_len - len(l))

    # --- PROCESS 2D STATIC EDGES -> NODES ---
    f_2d_edge = ['face_length', 'length', 'slope']#'relative_position_x', 'relative_position_y',
    for dirc, node_col, mx in [("from", "from_node", max_e_2d_from), ("to", "to_node", max_e_2d_to)]:
        gb = edges_2d_static.groupby(node_col)
        for feat in f_2d_edge:
            col_name = f"{feat}_{dirc}_node"
            res = gb[feat].apply(lambda x: pad_list(list(x), mx)).reset_index()
            res = res.rename(columns={node_col: "node_idx", feat: col_name})
            nodes_2d_static = nodes_2d_static.merge(res, on="node_idx", how="left")
            nodes_2d_static[col_name] = nodes_2d_static[col_name].apply(lambda x: x if isinstance(x, list) else [0.0]*mx)

    # --- PROCESS 1D STATIC EDGES -> NODES ---
    f_1d_edge = ['length', 'diameter', 'shape', 'roughness', 'slope']#'relative_position_x', 'relative_position_y',
    for dirc, node_col, mx in [("from", "from_node", max_e_1d_from), ("to", "to_node", max_e_1d_to)]:
        gb = edges_1d_static.groupby(node_col)
        for feat in f_1d_edge:
            col_name = f"{feat}_{dirc}_node"
            res = gb[feat].apply(lambda x: pad_list(list(x), mx)).reset_index()
            res = res.rename(columns={node_col: "node_idx", feat: col_name})
            nodes_1d_static = nodes_1d_static.merge(res, on="node_idx", how="left")
            nodes_1d_static[col_name] = nodes_1d_static[col_name].apply(lambda x: x if isinstance(x, list) else [0.0]*mx)

    # Final result dictionary for static features
    static_sample_1d = {}
    static_sample_2d = {}

    # Reshape 2D Node Statics
    f2d_node_base = ["area", "roughness", 'min_elevation', 'elevation', 'aspect', 'curvature', 'flow_accumulation']#['position_x', 'position_y', 'area', 'roughness', 'min_elevation', 'elevation', 'aspect', 'curvature', 'flow_accumulation']
    for f in f2d_node_base:
        static_sample_2d[f] = nodes_2d_static[f].values.astype(np.float32).squeeze()[...,None]

    # Reshape 2D Edge Statics (mapped to nodes)
    for f in f_2d_edge:
        for dirc, mx in [("from", max_e_2d_from), ("to", max_e_2d_to)]:
            col = f"{f}_{dirc}_node"
            static_sample_2d[col] = np.array(nodes_2d_static[col].tolist()).reshape(-1, mx).astype(np.float32)

    # Reshape 1D Node Statics
    f1d_node_base = ['depth', 'invert_elevation', 'surface_elevation', 'base_area']#['position_x', 'position_y', 'depth', 'invert_elevation', 'surface_elevation', 'base_area']
    for f in f1d_node_base:
        static_sample_1d[f] = nodes_1d_static[f].values.astype(np.float32).squeeze()[...,None]

    # Reshape 1D Edge Statics (mapped to nodes)
    for f in f_1d_edge:
        for dirc, mx in [("from", max_e_1d_from), ("to", max_e_1d_to)]:
            col = f"{f}_{dirc}_node"
            static_sample_1d[col] = np.array(nodes_1d_static[col].tolist()).reshape(-1, mx).astype(np.float32)

    num_1d_node = nodes_1d_static.node_idx.unique().size
    num_2d_node = nodes_2d_static.node_idx.unique().size
    indexer_1d = np.full((num_1d_node, 1), -1, dtype=np.int32)
    indexer_2d = np.full((num_2d_node, 1), -1, dtype=np.int32)

    indexer_1d[connection['node_1d'].values] = connection['node_2d'].values.reshape(-1, 1)
    indexer_2d[connection['node_2d'].values] = connection['node_1d'].values.reshape(-1, 1)
   
    edge_from_node_1d = np.full((num_1d_node, max_e_1d_from), -1, dtype=np.int32)
    edge_to_node_1d = np.full((num_1d_node, max_e_1d_to), -1, dtype=np.int32)

    edge_from_node_2d = np.full((num_2d_node, max_e_2d_from), -1, dtype=np.int32)
    edge_to_node_2d = np.full((num_2d_node, max_e_2d_to), -1, dtype=np.int32)

    # Counters to track the next available column for each node
    counts_from_1d = np.zeros(num_1d_node, dtype=np.int32)
    counts_to_1d = np.zeros(num_1d_node, dtype=np.int32)

    # 2. Populate 1D - Strict Node-to-Node mapping
    for _, row in edge_index_1d.iterrows():
        u, v = int(row['from_node']), int(row['to_node'])

        # edge_from_node_1d[u] stores nodes that u points TO
        edge_from_node_1d[u, counts_from_1d[u]] = v
        counts_from_1d[u] += 1

        # edge_to_node_1d[v] stores nodes that point TO v
        edge_to_node_1d[v, counts_to_1d[v]] = u
        counts_to_1d[v] += 1

    # 3. Populate 2D - Strict Node-to-Node mapping
    counts_from_2d = np.zeros(num_2d_node, dtype=np.int32)
    counts_to_2d = np.zeros(num_2d_node, dtype=np.int32)

    for _, row in edge_index_2d.iterrows():
        u, v = int(row['from_node']), int(row['to_node'])

        edge_from_node_2d[u, counts_from_2d[u]] = v
        counts_from_2d[u] += 1

        edge_to_node_2d[v, counts_to_2d[v]] = u
        counts_to_2d[v] += 1
        
    release_node_idx_1d = np.where(static_sample_1d["base_area"]==0)[0].reshape(-1, 1)
    return {
        "node_1d_static": static_sample_1d,
        "node_2d_static": static_sample_2d,
        "edge_connection":{
            "edge_from_node_1d": edge_from_node_1d,
            "edge_to_node_1d": edge_to_node_1d,
            "edge_from_node_2d": edge_from_node_2d,
            "edge_to_node_2d": edge_to_node_2d,
            "release_node_idx_1d": release_node_idx_1d,
        },
        "connection_1d2d": {"coupling_idx_1d":indexer_1d, "coupling_idx_2d": indexer_2d}
    }

In [ ]:
%%writefile hydrology_layers.py

@keras.saving.register_keras_serializable(package="Hydrology")
class StaticEncoder(layers.Layer):
    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model

        
        self.project_1d = layers.Dense(
            self.d_model,
            activation='swish',
            kernel_initializer='glorot_uniform'
        )
        self.project_2d = layers.Dense(
            self.d_model,
            activation='swish',
            kernel_initializer='glorot_uniform'
        )

    def call(self, inputs, training=False):
        node_1d_s, node_2d_s, base_area_mask, connection_to_r_node_1d = inputs
        s_1d = ops.concatenate([
            node_1d_s["depth"],
            node_1d_s["base_area"],
            node_1d_s["diameter_to_node"],
            node_1d_s["diameter_from_node"]*-1,
            node_1d_s["slope_to_node"],
            node_1d_s["slope_from_node"]*-1,
            node_1d_s["roughness_to_node"],
            node_1d_s["roughness_from_node"]*-1,

        ], -1)
        s_2d = ops.concatenate([
            node_2d_s["face_length_to_node"],
            node_2d_s["face_length_from_node"]*-1,
            node_2d_s["length_to_node"],
            node_2d_s["length_from_node"]*-1,
            node_2d_s["slope_to_node"],
            node_2d_s["slope_from_node"]*-1,
            node_2d_s["area"],
            node_2d_s["roughness"],
            #node_2d_s["elevation"],
            node_2d_s["aspect"],
            node_2d_s["curvature"],
            node_2d_s["flow_accumulation"],

        ], -1)



        s_1d_proj = self.project_1d(s_1d)  # (batch, nodes, d_model)
        s_2d_proj = self.project_2d(s_2d)
        return s_1d_proj, s_2d_proj
        
    def get_config(self):
        cfg = super().get_config()
        cfg.update(d_model=self.d_model)
        return cfg

    @classmethod
    def from_config(cls, config):
        return cls(**config)

@keras.saving.register_keras_serializable(package="Hydrology")
class CustomLSTMCell(keras.Layer):
    """
    Single LSTM cell — computes one timestep.
    """
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        input_dim = input_shape[-1]

        # Keras stores all gate weights concatenated as (input_dim, units*4)
        # Gate order: i, f, c, o  (same as Keras default)
        self.W = self.add_weight(
            shape=(input_dim, self.units * 4),
            initializer='glorot_uniform',
            name='W'
        )
        self.U = self.add_weight(
            shape=(self.units, self.units * 4),
            initializer='orthogonal',
            name='U'
        )
        self.b = self.add_weight(
            shape=(self.units * 4,),
            initializer='zeros',
            name='b'
        )
        # Initialize forget gate bias to 1 for better gradient flow
        b_init = np.ones((self.units * 4,), dtype=self.b.dtype)
        b_init[self.units:self.units*2] = -0.01  # forget gate is second slice
        self.b.assign(b_init)

        self.built = True

    def call(self, x, h, c):
        """
        x: (b, input_dim)
        h: (b, units)
        c: (b, units)
        returns: h_new (b, units), c_new (b, units)
        """
        # All gates in one matmul — efficient
        gates = x @ self.W + h @ self.U + self.b  # (b, units*4)

        u = self.units
        i = jax.nn.sigmoid(gates[:, 0*u:1*u])   # input gate
        f = jax.nn.sigmoid(gates[:, 1*u:2*u])   # forget gate
        g = jnp.tanh(gates[:, 2*u:3*u])         # cell gate
        o = jax.nn.sigmoid(gates[:, 3*u:4*u])   # output gate

        c_new = f * c + i * g
        h_new = o * jnp.tanh(c_new)

        return h_new, c_new
    def get_config(self):
        cfg = super().get_config()
        cfg.update(units=self.units)
        return cfg

    @classmethod
    def from_config(cls, config):
        return cls(**config)
        
@keras.saving.register_keras_serializable(package="Hydrology")
class CustomLSTM(keras.Layer):
    """
    Equivalent to keras.layers.LSTM(units, return_sequences=True).
    """
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.cell = CustomLSTMCell(units)
        self.project = layers.Dense(units)
        self.ln = layers.LayerNormalization()
       
    def call(self, inputs, initial_state=None, training=False):
        """
        inputs: (x, neighbor_idx, coupling_idx)
        x: (b*node, t, input_dim)
        initial_state: [h0, c0] each (b*nodes,units), or None
        returns: (b*node, t, units)
        """
        x, neighbor_idx, coupling_idx = inputs
        from_node_idx, to_node_idx = neighbor_idx
        b, t, _ = ops.shape(x)
        #b, n, _ = ops.shape(coupling_idx)
        if initial_state is None:
            h = jnp.zeros((b, self.units), dtype=x.dtype)
            c = jnp.zeros((b, self.units), dtype=x.dtype)
        else:
            h, c = initial_state

        # scan expects (t, b, input_dim)
        x_T = jnp.transpose(x, (1, 0, 2))  # (t, b, input_dim)

        def step(carry, x_t):
            h, c = carry
            h_new, c_new = self.cell(x_t, h, c)
            return (h_new, c_new), h_new

        (h_final, c_final), h_seq = jax.lax.scan(step, (h, c), x_T)

        # h_seq: (t, b, units) -> (b, t, units)
        return jnp.transpose(h_seq, (1, 0, 2))
    def get_config(self):
        cfg = super().get_config()
        cfg.update(units=self.units)
        return cfg

    @classmethod
    def from_config(cls, config):
        return cls(**config)


class NodeSpecificProjector(keras.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
    def build(self, input_shape):
        num_nodes = input_shape[2]
        embed_dim = input_shape[3]

        # Define a custom internal initializer function
        def synced_init(shape, dtype=None):
            # 1. Generate the base weights (the shared logic)
            base_init = keras.initializers.VarianceScaling(
                scale=0.01, mode="fan_in", distribution="truncated_normal"
            )
            base_weights = base_init(shape=(embed_dim, self.units))

            # 2. Broadcast inside the initializer context
            return ops.broadcast_to(base_weights, (num_nodes, embed_dim, self.units))

        self.node_weights = self.add_weight(
            name="node_weights",
            shape=(num_nodes, embed_dim, self.units),
            initializer=synced_init,
            trainable=True
        )

        self.node_bias = self.add_weight(
            name="node_bias",
            shape=(num_nodes, self.units),
            initializer="zeros",
            trainable=True
        )

    def call(self, x):
        res = ops.einsum('blne,neo->blno', x, self.node_weights)
        return res + self.node_bias
        
    def get_config(self):
        cfg = super().get_config()
        cfg.update(units=self.units)
        return cfg

    @classmethod
    def from_config(cls, config):
        return cls(**config)

@keras.saving.register_keras_serializable(package="Hydrology")
class FloodModel(keras.Model):
    def __init__(self, cfg, **kwargs):
        super().__init__(**kwargs)
        self.cfg = cfg
        self.d_model = cfg['d_model']
        self.static_encoder = StaticEncoder(self.d_model)
        self.lstm_layers_1d = [CustomLSTM(self.d_model) for i in range(3)]
        self.lstm_layers_2d = [CustomLSTM(self.d_model) for i in range(2)]

        self.project_1d = layers.Dense(self.d_model, activation='tanh')
        self.project_2d = layers.Dense(self.d_model, activation='tanh')
       
        self.ln_1d = keras.layers.LayerNormalization(epsilon=1e-5)
        self.ln_2d = keras.layers.LayerNormalization(epsilon=1e-5)
        
        self.inp_expand_1d = NodeSpecificProjector(128)
        self.inp_expand_2d = NodeSpecificProjector(64)
        self.decoder_1d = NodeSpecificProjector(1)
        self.decoder_2d = NodeSpecificProjector(1)

    def call(self, inputs, training=False):
       
        node_1d_static, node_2d_static = inputs["node_1d_static"], inputs["node_2d_static"]
        rain_2d = inputs["rain_2d"]
      
        s_idx_from_1d = ops.cast(inputs["edge_connection"]["edge_from_node_1d"], "int32")
        s_idx_to_1d = ops.cast(inputs["edge_connection"]["edge_to_node_1d"], "int32")
        s_idx_from_2d = ops.cast(inputs["edge_connection"]["edge_from_node_2d"], "int32")
        s_idx_to_2d = ops.cast(inputs["edge_connection"]["edge_to_node_2d"], "int32")
        release_node_idx_1d =  inputs["edge_connection"]["release_node_idx_1d"]
        c1d_idx = ops.cast(inputs["connection_1d2d"]["coupling_idx_1d"], "int32")
        c2d_idx = ops.cast(inputs["connection_1d2d"]["coupling_idx_2d"], "int32")

        base_area = node_1d_static["base_area"]
        base_area_mask = ops.cast(base_area!=0, 'float32')
        connection_to_r_node_1d = ops.take(base_area_mask, s_idx_from_1d[0], axis=1)[...,0]

        stat_1d, stat_2d = self.static_encoder([node_1d_static, node_2d_static, base_area_mask, connection_to_r_node_1d], training=training)

        b, n1d, d = ops.shape(stat_1d)
        b, t, n2d, _ = ops.shape(rain_2d[:,10:])

        net_diameter = node_1d_static["diameter_to_node"].sum(-1,keepdims=True) - node_1d_static["diameter_from_node"]
        area = node_2d_static["area"]
        minimum_elevation = node_2d_static["min_elevation"]
        elevation = node_2d_static["elevation"]
        minimum_elevation = ops.where(ops.isfinite(minimum_elevation), minimum_elevation, elevation)
        flow_to_node_2d = inputs["node_2d"]["flow_to_node_2d"]
        flow_from_node_2d = inputs["node_2d"]["velocity_from_node_2d"] * -1
        vel_to_node_2d = inputs["node_2d"]["velocity_to_node_2d"]
        vel_from_node_2d = inputs["node_2d"]["flow_from_node_2d"] * -1
        water_level = inputs["node_2d"]["water_level"] - minimum_elevation[:,None]
        zero_node = ops.zeros((b, 10, 1, 1))
        water_level = ops.concatenate([water_level, zero_node], axis=2)
        wl_neighbors1_2d = ops.take(water_level, s_idx_from_2d[0], axis=2)[...,0]
        wl_neighbors2_2d = ops.take(water_level, s_idx_to_2d[0], axis=2)[...,0]

        delta_2d = water_level[:,1:] - water_level[:,:-1]
        delta_2d = ops.concatenate([ops.zeros_like(delta_2d[:,:1]), delta_2d], axis=1)

        delta_neighbors1_2d = ops.take(delta_2d, s_idx_from_2d[0], axis=2)[...,0]
        delta_neighbors2_2d = ops.take(delta_2d, s_idx_to_2d[0], axis=2)[...,0]

        delta_2d = delta_2d[:,:,:-1]
        water_level = water_level[:,:,:-1]

        #1d
        invert_elevation = node_1d_static["invert_elevation"]
        water_level_1d = inputs["node_1d"]["1d_water_level"]
        wl_from_invert = water_level_1d - invert_elevation[:,None]
        wl_from_invert = wl_from_invert
        wl_from_surface = node_1d_static["surface_elevation"][:,None] - water_level_1d

        delta = water_level_1d[:,1:] - water_level_1d[:,:-1]
        delta = ops.concatenate([ops.zeros_like(delta[:,:1]), delta], axis=1)
        zero_node = ops.zeros((b, 10, 1, 1))
        delta = ops.concatenate([delta, zero_node], axis=2)
        delta_neighbors1_1d = ops.take(delta, s_idx_from_1d[0], axis=2)[...,0]
        delta_neighbors2_1d = ops.take(delta, s_idx_to_1d[0], axis=2)[...,0]
        delta_release_node_1d = ops.take(delta, ops.repeat(release_node_idx_1d[0,None,:,0],n1d,axis=0), axis=2)[...,0]

        wl_from_invert = ops.concatenate([wl_from_invert, zero_node], axis=2)
        wl_from_surface = ops.concatenate([wl_from_surface, zero_node], axis=2)
        wl_neighbors1_1d = ops.take(wl_from_invert, s_idx_from_1d[0], axis=2)[...,0]
        wl_neighbors2_1d = ops.take(wl_from_invert, s_idx_to_1d[0], axis=2)[...,0]
        wl_2dneightbors_1d = ops.take(water_level[:,], c1d_idx[0], axis=2)[...,0]
        flow_to_node = inputs["node_1d"]["flow_to_node_1d"]
        flow_from_node = inputs["node_1d"]["flow_from_node_1d"] * -1
        vel_to_node_1d = inputs["node_1d"]["velocity_to_node_1d"]
        vel_from_node_1d = inputs["node_1d"]["flow_from_node_1d"] * -1
        inlet_flow = inputs["node_1d"]["inlet_flow"]

        delta = delta[:,:,:-1]
        wl_from_invert = wl_from_invert[:,:,:-1]
        wl_from_surface = wl_from_surface[:,:,:-1]


        x_1d = rain_2d[:,10:,:n1d]
        x_2d = rain_2d[:,10:,:n2d]

        x_1d = self.inp_expand_1d(x_1d)
        x_2d = self.inp_expand_2d(x_2d)

        x_1d = ops.reshape(ops.transpose(x_1d, (0,2,1,3)), (b*n1d, t, -1))
        initial_state = ops.concatenate([
            flow_to_node, flow_from_node, inlet_flow, wl_from_invert, wl_from_surface,
            wl_neighbors1_1d, wl_neighbors2_1d, delta, delta_neighbors1_1d, delta_neighbors2_1d],
                                        -1)[:,-1]
       
        initial_state = self.project_1d(initial_state)
        initial_state = ops.reshape(initial_state , (b*n1d, -1))
        c = ops.reshape(stat_1d, (b*n1d, -1))
        initial_state = (initial_state, c)
        for layer in self.lstm_layers_1d:
            x_1d = layer((x_1d, (s_idx_to_1d, s_idx_from_1d), c1d_idx), initial_state=initial_state)

        
        x_2d = ops.reshape(ops.transpose(x_2d, (0,2,1,3)), (b*n2d, t, -1))
        initial_state = ops.concatenate([
            flow_to_node_2d, flow_to_node_2d, water_level, wl_neighbors1_2d, wl_neighbors2_2d,
            delta_2d, delta_neighbors1_2d, delta_neighbors2_2d], -1)[:,-1]
        initial_state = self.project_2d(
            ops.reshape(initial_state , (b*n2d, -1))
            )
        c = ops.reshape(stat_2d, (b*n2d, -1))
        initial_state = (initial_state, c)
        for layer in self.lstm_layers_2d:
            x_2d = layer((x_2d, (s_idx_to_2d, s_idx_from_2d), c2d_idx), initial_state=initial_state)

        x_1d = ops.transpose(ops.reshape(x_1d, (b, n1d, t, -1)), (0, 2, 1, 3))
        x_2d = ops.transpose(ops.reshape(x_2d, (b, n2d, t, -1)), (0, 2, 1, 3))

        base_area = ops.where(base_area==0, 1, base_area)[:,None]
        area = ops.where(area<=0, 1, area)[:,None]
        return {
            "out_1d": self.decoder_1d(self.ln_1d(x_1d)) / base_area,
            "out_2d": self.decoder_2d(self.ln_2d(x_2d)) / area
        }

    def get_config(self):
        config = super().get_config()
        config.update({"cfg": self.cfg})
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)
    def compute_loss_and_updates(
        self,
        trainable_variables,
        non_trainable_variables,
        metrics_variables,
        x,
        y,
        sample_weight,
        training=False,
    ):
        y_pred, non_trainable_variables = self.stateless_call(
            trainable_variables,
            non_trainable_variables,
            x,
            training=training,
        )
        loss = self.loss_fn(y, y_pred)
        return loss, (y_pred, non_trainable_variables, metrics_variables)
   
    def train_step(self, state, data):
        (
            trainable_variables,
            non_trainable_variables,
            optimizer_variables,
            metrics_variables,
        ) = state
        x, y, sample_weight = keras.utils.unpack_x_y_sample_weight(data)

        # Get the gradient function.
        grad_fn = jax.value_and_grad(self.compute_loss_and_updates, has_aux=True)

        # Compute the gradients.
        (loss, (y_pred, non_trainable_variables, metrics_variables)), grads = grad_fn(
            trainable_variables,
            non_trainable_variables,
            metrics_variables,
            x,
            y,
            sample_weight,
            training=True,
        )
       
        trainable_variables, optimizer_variables = self.optimizer.stateless_apply(
            optimizer_variables, grads, trainable_variables
        )

        trainable_variables = [
            var - (self.wd_schedule(optimizer_variables[0]) * self.weight_decay * var) for var in trainable_variables
        ]
        # Return metric logs and updated state variables.
        state = (
            trainable_variables,
            non_trainable_variables,
            optimizer_variables,
            metrics_variables,
        )
        return loss, y_pred, state
    
    def test_step(self, state, data):
        # Unpack the data.
        x, y, sample_weight = keras.utils.unpack_x_y_sample_weight(data)
        (
            trainable_variables,
            non_trainable_variables,
            metrics_variables,
        ) = state

        # Compute predictions and loss.
        y_pred, non_trainable_variables = self.stateless_call(
            trainable_variables,
            non_trainable_variables,
            x,
            training=False,
        )
        loss = self.loss_fn(y, y_pred)

        # Update metrics.
        new_metrics_vars = []

        # Return metric logs and updated state variables.
        state = (
            trainable_variables,
            non_trainable_variables,
            metrics_variables,
        )
        return loss, y_pred, state

In [ ]:
exec(open("./hydrology_layers.py", 'r').read())

In [ ]:
def get_replicated_train_state(model, optimizer, devices):
    mesh = Mesh(devices, axis_names=("_",))
    replicated = NamedSharding(mesh, P())

    trainable_vars = jax.device_put([jnp.array(w) for w in model.trainable_variables], replicated)
    non_trainable_vars = jax.device_put([jnp.array(w) for w in model.non_trainable_variables], replicated)
    optimizer_vars =jax.device_put([jnp.array(w) for w in optimizer.variables], replicated)
    metrics_variables = jax.device_put([jnp.array(w) for w in model.metrics_variables], replicated)

    return (trainable_vars, non_trainable_vars, optimizer_vars, metrics_variables)


In [ ]:
def std_rmse(y_true, y_pred, std_dev):
    """
    Calculates the RMSE for a domain, averaged across all nodes.
    Used here to get the domain-specific part of the event scalar.
    """
    se_standardized = ((y_true - y_pred) / std_dev) ** 2
    mse_per_node = np.mean(se_standardized, axis=0)
    rmse_per_node = np.sqrt(mse_per_node)
    return np.mean(rmse_per_node), rmse_per_node

def calculate_score(y_true_list, y_pred_dict):
    # 1. Use YOUR flattening logic - this is correct for your data structure
    true_1d_samples = [s for b in y_true_list for s in b["water_level_1d_target"]]
    base_area = y_true_list[0]["node_1d_static"]["base_area"][0]
    base_area = np.where(base_area==0, 1, base_area)[None,:,0]
    invert_elevatin =  y_true_list[0]["node_1d_static"]["invert_elevation"][0][None,:,0]
    true_2d_samples = [s for b in y_true_list for s in b["water_volume_2d_target"]]
    true_2d_samples_level = [s for b in y_true_list for s in b["water_level_2d_target"]]

    # Restoring your prediction flattening logic
    preds_1d = [s for b in y_pred_dict for s in b["out_1d"]]
    preds_2d = [s for b in y_pred_dict for s in b["out_2d"]]

    event_scores = []
    event_scores_1d = []
    event_scores_2d = []
    event_srmse_per_node_1d = []
    event_srmse_per_node_2d = []
    for i in range(len(true_1d_samples)):
        # Squeeze to (Timestep, Nodes)
        t1 = np.squeeze(true_1d_samples[i])
        t2 = np.squeeze(true_2d_samples[i])

        p1_deltas = np.squeeze(preds_1d[i]) #/ base_area
        p2_deltas = np.squeeze(preds_2d[i])

        # Time masks
        v1 = t1[10:, 0] != -1.0
        v2 = t2[10:, 0] != -1.0

        # 1D Reconstruction
        y_p_1d = (t1[9, :][np.newaxis, :] + np.cumsum(p1_deltas, axis=0))[v1]
        y_p_1d = y_p_1d.clip(min_wl_pn_1d, max_wl_pn_1d)
        score_1d, srmse_per_node_1d = std_rmse(t1[10:][v1], y_p_1d, training_cfg["std_dev_1d"])
        event_srmse_per_node_1d.append(srmse_per_node_1d)
        # 2D Reconstruction

        y_p_2d = (true_2d_samples_level[i][9, :, 0][np.newaxis, :] + np.cumsum(p2_deltas, axis=0))[v2]
        y_p_2d = y_p_2d.clip(min_wl_pn_2d, max_wl_pn_2d)
        score_2d, srmse_per_node_2d = std_rmse(true_2d_samples_level[i][10:][v2], y_p_2d[...,None], training_cfg["std_dev_2d"])
        event_srmse_per_node_2d.append(srmse_per_node_2d)

        # Per-event scalar
        event_scores.append((score_1d + score_2d) / 2.0)
        event_scores_1d.append(score_1d)
        event_scores_2d.append(score_2d)
    event_srmse_per_node_1d = np.stack(event_srmse_per_node_1d, 0).mean(0)
    event_srmse_per_node_2d = np.stack(event_srmse_per_node_2d, 0).mean(0)
    return np.mean(event_scores), np.mean(event_scores_1d), np.mean(event_scores_2d), event_srmse_per_node_1d, event_srmse_per_node_2d

In [ ]:
def train_model(
    model,
    train_data,
    val_data,
    epochs,
    lookahead_sync_period=6,
    lookahead_alpha=0.5,
):
    # ---- Devices & sharding ----
    devices = jax.local_devices()
    num_devices = len(devices)
    print(f"Running on {num_devices} devices")

    device_mesh = mesh_utils.create_device_mesh((num_devices,))
    data_mesh = Mesh(device_mesh, axis_names=("batch",))
    data_sharding = NamedSharding(data_mesh, P("batch"))

    model.optimizer.build(model.trainable_variables)

    # ---- Initial state ----
    train_state = get_replicated_train_state(model, model.optimizer, device_mesh)

    # ---- Lookahead: initialise slow weights from current trainable vars ----
    # train_state = (trainable_vars, non_trainable_vars, optimizer_vars, metrics_vars)
    # slow_weights mirrors trainable_vars only
    slow_weights = train_state[0]

    global_step = 0
    train_step = jax.jit(model.train_step)
    test_step  = jax.jit(model.test_step)

    # ---- Training loop ----
    for epoch in range(epochs):
        epoch_start = time.time()

        batch_losses = []
        y_pred = []
        y_true = []
        for step, (x, y) in enumerate(train_data):
            x = jax.device_put(x, data_sharding)
            y = jax.device_put(y, data_sharding)
            y_true.append(y)

            loss, pred, train_state = train_step(train_state, (x, y))

            y_pred.append(pred)
            batch_losses.append(float(loss))
            global_step += 1

            # ---- Lookahead sync ----
            if global_step % lookahead_sync_period == 0:
                trainable_vars, non_trainable_vars, optimizer_vars, metrics_variables = train_state

                # Interpolate: slow += alpha * (fast - slow), then reset fast = slow
                new_slow = [
                    slow + lookahead_alpha * (fast - slow)
                    for slow, fast in zip(slow_weights, trainable_vars)
                ]
                slow_weights = new_slow
                train_state = slow_weights, non_trainable_vars, optimizer_vars, metrics_variables
        epoch_loss     = float(np.mean(batch_losses))
        train_data.on_epoch_end()
        if epoch % 500 == 0:
            score, score_1d, score_2d, srmse_per_node_1d, srmse_per_node_2d = calculate_score(y_true, y_pred)

        # ---- Validation with slow weights ----
        trainable_vars, non_trainable_vars, optimizer_vars, metrics_variables = train_state
        if val_data is not None:

            test_state = trainable_vars, non_trainable_vars, metrics_variables

            val_batch_losses = []
            val_y_pred = []
            val_y_true = []
            for step, (x, y) in enumerate(val_data):
                x = jax.device_put(x, data_sharding)
                y = jax.device_put(y, data_sharding)
                val_y_true.append(y)

                val_loss, pred, test_state = test_step(test_state, (x, y))

                val_y_pred.append(pred)
                val_batch_losses.append(float(val_loss))

            val_epoch_loss = float(np.mean(val_batch_losses))
            if epoch % 500 == 0:
                val_score, val_score_1d, val_score_2d, val_srmse_per_node_1d, val_srmse_per_node_2d = calculate_score(val_y_true, val_y_pred)
        else:
            val_epoch_loss = epoch_loss
            val_score = score
            val_score_1d = score_1d
            val_score_2d = score_2d
            val_srmse_per_node_1d = srmse_per_node_1d
            val_y_true, val_y_pred = y_true, y_pred
        epoch_time     = time.time() - epoch_start

        print(
            f"Epoch {epoch + 1}/{epochs} "
            f"- loss: {epoch_loss:.6f}, "
            f"score: {score:.8f} "
            f"score_1d: {score_1d:.8f} "
            f"score_2d: {score_2d:.8f} "
            f"- val_loss: {val_epoch_loss:.6f}, "
            f"val_score: {val_score:.8f} "
            f"val_score_1d: {val_score_1d:.8f} "
            f"val_score_2d: {val_score_2d:.8f} "
            f"- time: {epoch_time:.2f}s"
        )
        train_state = trainable_vars, non_trainable_vars, optimizer_vars, metrics_variables

    for var, val in zip(model.trainable_variables, trainable_vars):
        var.assign(val)

    for var, val in zip(model.non_trainable_variables, non_trainable_vars):
        var.assign(val)

    return model, val_y_true, val_y_pred

In [ ]:
def build_model(model_cfg, training_cfg):
   
    lr_schedule = optimizers.schedules.CosineDecay(
        initial_learning_rate=training_cfg['initial_lr'],
        decay_steps=training_cfg['decay_steps'],
        alpha=training_cfg['alpha']
    )

    optimizer = optimizers.Nadam(
        learning_rate=lr_schedule,
        beta_1=training_cfg['beta_1'],
        beta_2=training_cfg['beta_2'],
        epsilon=training_cfg['epsilon'],
    )

    # 3. Model Initialization
    model = FloodModel(model_cfg)

    loss_fn = training_cfg['loss_fn'](**training_cfg["loss_fn_args"])
    model.loss_fn = loss_fn
    model.weight_decay = training_cfg["weight_decay"]
    model.wd_schedule = lr_schedule

    # 4. Compilation
    # Loss and metrics are defined in training_cfg
    model.compile(
        optimizer=optimizer,
        loss="mse"
    )
    
    return model

In [ ]:
def split_indices_into_folds(num_events, num_folds, seed):
    """
    Creates k-fold splits of indices.
    """
    indices = np.arange(num_events)
    #np.random.seed(seed)
    #np.random.shuffle(indices)

    # Split the array into num_folds subarrays
    return np.array_split(indices, num_folds)

def get_fold_indices(folds, val_fold_index, device_count=8):
    """
    Returns train_idx and val_idx, ensuring val_idx is a multiple of device_count
    by taking samples from train_idx.
    """
    val_idx = folds[val_fold_index]
    train_idx = np.concatenate([f for i, f in enumerate(folds) if i != val_fold_index])

    # Calculate how many extra samples we need to reach a multiple of device_count
    remainder = len(val_idx) % device_count
    if remainder != 0:
        num_to_borrow = device_count - remainder

        # Take from the start of training indices
        borrowed_idx = train_idx[:num_to_borrow]
        train_idx = train_idx[num_to_borrow:]

        # Add to validation
        val_idx = np.concatenate([val_idx, borrowed_idx])

    return train_idx, val_idx

In [ ]:
%%writefile flood_dataloader.py
class FloodDataLoader(keras.utils.PyDataset):
    def __init__(self, data, indices, batch_size, max_timesteps, data_path,
                 train=False, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.data = data # This is a list of paths or preloaded dicts
        self.indices = indices
        self.batch_size = batch_size
        self.max_timesteps = max_timesteps
        self.train = train
        self.shuffle = shuffle
        self.input_window = 10
        self.static_data = load_static_feature(data_path)
    def __len__(self):
        return int(np.floor(len(self.indices)/self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            random.shuffle(self.indices)

    def _pad_and_slice(self, event_dict):
        """Explicitly slices and pads each feature by name."""
        total_steps = event_dict["rainfall"].shape[0]

        # Temporal slicing logic
        if self.train and total_steps > self.input_window + 1:
            if np.random.uniform(0, 1)<0.8:
                start_t = random.randint(0, 20)
            else:
                start_t = 0

        else:
            start_t = 0

        end_in = start_t + self.input_window

        # 1. Gather 2D Node Inputs (Explicitly)
        # Shape: (10, num_nodes) or (10, num_nodes, max_edges)
        inputs_2d = {
            #"rainfall": event_dict["rainfall"][start_t:end_in],
            "water_level": event_dict["water_level"][start_t:end_in],
            "water_volume": event_dict["water_volume"][start_t:end_in],
            "flow_from_node_2d": event_dict["flow_from_node_2d"][start_t:end_in],
            "velocity_from_node_2d": event_dict["velocity_from_node_2d"][start_t:end_in],
            "flow_to_node_2d": event_dict["flow_to_node_2d"][start_t:end_in],
            "velocity_to_node_2d": event_dict["velocity_to_node_2d"][start_t:end_in],
        }

        # 2. Gather 1D Node Inputs (Explicitly)
        inputs_1d = {
            "1d_water_level": event_dict["1d_water_level"][start_t:end_in],
            "inlet_flow": event_dict["inlet_flow"][start_t:end_in],
            "flow_from_node_1d": event_dict["flow_from_node_1d"][start_t:end_in],
            "velocity_from_node_1d": event_dict["velocity_from_node_1d"][start_t:end_in],
            "flow_to_node_1d": event_dict["flow_to_node_1d"][start_t:end_in],
            "velocity_to_node_1d": event_dict["velocity_to_node_1d"][start_t:end_in],
        }

        # 3. Gather Targets (Explicitly)
        targets = {
            "water_volume_2d_target": event_dict["water_volume"][start_t:],
            "water_level_2d_target": event_dict["water_level"][start_t:],
            "water_level_1d_target": event_dict["1d_water_level"][start_t:]
        }

        # Helper function for explicit padding
        def apply_padding(data_dict, max_t, pad_value=0):
            for k, v in data_dict.items():
                current_t = v.shape[0]
                if current_t < max_t:
                    # Pad the first dimension (Time)
                    pad_dims = [(0, max_t - current_t)] + [(0, 0)] * (v.ndim - 1)

                    data_dict[k] = np.pad(v, pad_dims, mode='constant', constant_values=pad_value)

                data_dict[k] = data_dict[k][:max_t]

            return data_dict

        # Apply padding to all groups
        inputs_2d = apply_padding(inputs_2d, self.input_window) # Input is always 10
        inputs_1d = apply_padding(inputs_1d, self.input_window)

        # Targets are padded to the remaining length (max_timesteps - 10)
        if self.max_timesteps:
            target_max_t = self.max_timesteps #- self.input_window
        else:
            target_max_t = event_dict["rainfall"].shape[0]
        targets = apply_padding(targets, target_max_t, pad_value=-1)
        rainfall = {"rainfall": event_dict["rainfall"][start_t:]}
        rainfall = apply_padding(rainfall, target_max_t)
        return {"1d_nodes": inputs_1d, "2d_nodes": inputs_2d}, rainfall["rainfall"], targets

    def __getitem__(self, index):
       
        start_idx = index * self.batch_size
        end_idx = (index + 1) * self.batch_size
        batch_indices = self.indices[start_idx:end_idx]
        raw_batch = [self.data[i] for i in batch_indices]
        
        # Apply padding, slicing, and structuring
        processed_inputs = []
        rain_2d =  []
        processed_targets = []
        t1 = time.time()
        for event in raw_batch:
            inp, rainfall, tar = self._pad_and_slice(event)
            processed_inputs.append(inp)
            rain_2d.append(rainfall.squeeze()[...,None])
            processed_targets.append(tar)

        # 1. Explicitly batch 1D Node Features
        final_1d = {
            "1d_water_level": np.stack([pt["1d_nodes"]["1d_water_level"].squeeze()[...,None] for pt in processed_inputs]),
            "inlet_flow": np.stack([pt["1d_nodes"]["inlet_flow"].squeeze()[...,None] for pt in processed_inputs]),
            "flow_from_node_1d": np.stack([pt["1d_nodes"]["flow_from_node_1d"] for pt in processed_inputs]),
            "velocity_from_node_1d": np.stack([pt["1d_nodes"]["velocity_from_node_1d"] for pt in processed_inputs]),
            "flow_to_node_1d": np.stack([pt["1d_nodes"]["flow_to_node_1d"] for pt in processed_inputs]),
            "velocity_to_node_1d": np.stack([pt["1d_nodes"]["velocity_to_node_1d"] for pt in processed_inputs]),
        }

        # 2. Explicitly batch 2D Node Features
        final_2d = {
            "water_level": np.stack([pt["2d_nodes"]["water_level"].squeeze()[...,None] for pt in processed_inputs]),
            "water_volume": np.stack([pt["2d_nodes"]["water_volume"].squeeze()[...,None] for pt in processed_inputs]),
            "flow_from_node_2d": np.stack([pt["2d_nodes"]["flow_from_node_2d"] for pt in processed_inputs]),
            "velocity_from_node_2d": np.stack([pt["2d_nodes"]["velocity_from_node_2d"] for pt in processed_inputs]),
            "flow_to_node_2d": np.stack([pt["2d_nodes"]["flow_to_node_2d"] for pt in processed_inputs]),
            "velocity_to_node_2d": np.stack([pt["2d_nodes"]["velocity_to_node_2d"] for pt in processed_inputs]),
        }
        self.static_data["node_1d_static"]["water_level_range"] = max_wl_pn_1d - min_wl_pn_1d
        # 3. Explicitly batch Targets
        rain_2d = np.stack(rain_2d, 0)
        final_targets = {
            "water_volume_2d_target": np.stack([pt["water_volume_2d_target"].squeeze()[...,None] for pt in processed_targets]),
            "water_level_2d_target": np.stack([pt["water_level_2d_target"].squeeze()[...,None] for pt in processed_targets]),
            "water_level_1d_target": np.stack([pt["water_level_1d_target"].squeeze()[...,None] for pt in processed_targets]),
            "node_1d_static": {k:v[None,].repeat(self.batch_size, axis=0) for k,v in self.static_data["node_1d_static"].items()},
        }

        final_inputs = {
            "node_1d": final_1d,
            "node_1d_static": {k:v[None,].repeat(self.batch_size, axis=0) for k,v in self.static_data["node_1d_static"].items()},
            "node_2d": final_2d,
            "node_2d_static": {k:v[None,].repeat(self.batch_size, axis=0) for k,v in self.static_data["node_2d_static"].items()},
            "edge_connection": {k:v[None,].repeat(self.batch_size, axis=0) for k,v in self.static_data["edge_connection"].items()},
            "connection_1d2d": {k:v[None,].repeat(self.batch_size, axis=0) for k,v in self.static_data["connection_1d2d"].items()},
            "rain_2d": rain_2d,
        }

        return final_inputs, final_targets



In [ ]:
exec(open('./flood_dataloader.py', 'r').read())

In [ ]:
@keras.saving.register_keras_serializable(package="Hydrology")
class HydraulicDeltaLoss(keras.losses.Loss):
    def __init__(self, w_1d=1, w_2d=1, mask_val=-1, name="hydraulic_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.w_1d = w_1d
        self.w_2d = w_2d
        self.mask_val = mask_val

    def compute_masked_mse(self, y_true, y_pred):
        mask = ops.cast(ops.not_equal(y_true, self.mask_val), dtype=y_pred.dtype)[:,10:]

        target_future = y_true[:,1:] - y_true[:,:-1]
        target_future = target_future[:,9:] * mask
        y_pred = y_pred * mask
        
        squared_error = ops.square(target_future - y_pred)
        masked_error = squared_error * mask
        loss = masked_error.sum(1) / mask.sum(1)
        loss = loss.mean()
      
        return loss
    def compute_masked_mse1(self, y_true, y_pred, static_data):
        water_level_range_mask = ops.cast(static_data['water_level_range']!=0, 'float32')[:,None,:,None]
        mask = ops.cast(ops.not_equal(y_true, self.mask_val), dtype=y_pred.dtype)[:,10:]
        mask = mask * water_level_range_mask

        target = y_true[:,10:] * mask
        y_pred = y_true[:,9:10] + ops.cumsum(y_pred, axis=1)
        y_pred = y_pred * mask
        
        squared_error = ops.square(target - y_pred)
        masked_error = squared_error * mask
        loss = masked_error.sum(1) / (mask.sum(1)+1e-9)
        loss = loss.mean()

        return loss
    def call(self, y_true, y_pred):
        loss_1d = self.compute_masked_mse1(y_true['water_level_1d_target'], y_pred['out_1d'], y_true["node_1d_static"])
        loss_2d = self.compute_masked_mse(y_true['water_level_2d_target'], y_pred['out_2d'])

        return (self.w_1d * loss_1d) + (self.w_2d * loss_2d)

    def get_config(self):
        config = super().get_config()
        config.update({
            "w_1d": self.w_1d,
            "w_2d": self.w_2d,
            "mask_val": self.mask_val,
        })
        return config

In [ ]:
model_cfg = {
    "d_model": 128,
}
training_cfg = {
    "initial_lr": 2e-3,
    "alpha": 1e-4,
    "beta_1": 0.99,
    "beta_2": 0.999,
    "epsilon": 1e-7,
   
    "loss_fn": HydraulicDeltaLoss,
    "loss_fn_args": {
        "w_1d": 1,
        "w_2d": 1e+3,
    },
    "weight_decay":0,
    
    "batch_size":8,
    "num_epoch":6001,
    "std_dev_1d":3.191784,#16.877747,#
    "std_dev_2d":2.727131,#14.378797#
    "awp_start_epoch": 20,
    "awp_epsilon": 1e-2,
    "awp_eps_norm": 1e-5,
    "train_on_full_data":True,
}
data_cfg = {
    "train_forecast_steps": 400,
    "val_forecast_steps":400,
    "num_fold": 5,
    "seed": 000,
}

In [ ]:
mmap_mode = 'r'
if data_not_loaded:
    folds = split_indices_into_folds(len(events), data_cfg["num_fold"], data_cfg["seed"])
    events = [load_event_data(event, mmap_mode=mmap_mode) for event in tqdm(events)]
    data_not_loaded = False


max_wl_pn_1d = np.stack([event["1d_water_level"].max(0) for event in events], 0).max(0)
min_wl_pn_1d = np.stack([event["1d_water_level"].min(0) for event in events], 0).min(0)
max_wl_pn_2d = np.stack([event["water_level"].max(0) for event in events], 0).max(0)
min_wl_pn_2d = np.stack([event["water_level"].min(0) for event in events], 0).min(0)

np.save('max_wl_pn_1d.npy', max_wl_pn_1d)
np.save('min_wl_pn_1d.npy', min_wl_pn_1d)
np.save('max_wl_pn_2d.npy', max_wl_pn_2d)
np.save('min_wl_pn_2d.npy', min_wl_pn_2d)

In [ ]:
import math

for i in range(0, 2):
    print(f"\n--- Starting Fold {i + 1} / {data_cfg['num_fold']} ---")
    keras.backend.clear_session()
    set_seed(i)
    data_parallel = keras.distribution.DataParallel()
    keras.distribution.set_distribution(data_parallel)
    keras.mixed_precision.set_global_policy("mixed_bfloat16")
    
    train_idx, val_idx = get_fold_indices(folds, i)
    if training_cfg["train_on_full_data"]:
        train_idx = np.concatenate([val_idx, train_idx])
    
    print(f"num_val_data: {len(val_idx)}")
   
    train_ds = FloodDataLoader(
        events,
        train_idx,
        batch_size=training_cfg["batch_size"],
        max_timesteps=data_cfg["train_forecast_steps"]+10,
        data_path=data_path,
        train=True,
        shuffle=True,
    )
    dummy_ds = FloodDataLoader(
        events,
        train_idx,
        batch_size=training_cfg["batch_size"],
        max_timesteps=10+10,
        data_path=data_path,
        train=True,
        shuffle=True,
    )
    if training_cfg["train_on_full_data"]:
        val_ds = None
    else:
        val_ds = FloodDataLoader(
        events,
        val_idx,
        batch_size=training_cfg["batch_size"],
        max_timesteps=data_cfg["val_forecast_steps"]+10,
        data_path=data_path,
        train=False,
        shuffle=False,
    )
    for batch in dummy_ds:
        break

    steps_per_epoch = math.ceil(len(train_idx) / training_cfg["batch_size"])
    training_cfg["decay_steps"] = steps_per_epoch * training_cfg["num_epoch"]
    training_cfg["awp_start_step"] = steps_per_epoch * training_cfg["awp_start_epoch"]
   
    model = build_model(model_cfg, training_cfg)
    pred = model(batch[0])
    model.summary()
   
    model.save(f"model_{i}.keras")
    model.load_weights(f"./model_{i}.keras")
    
    score, score_1d, score_2d, score_1d_per_node, score_2d_per_node = calculate_score([batch[1]], [pred])
    print((score, score_1d, score_2d, score_1d_per_node.max(), score_2d_per_node.max()))

    model, y_true, y_pred = train_model(
        model,
        train_ds,
        val_ds,
        epochs=training_cfg["num_epoch"],
        lookahead_sync_period=6,
        lookahead_alpha=0.5
    )
    model.save(f"model_{i}.keras")
    #break